In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from typing import Dict, Any, List
import torch
import torch.nn.functional as F
import ray
import gymnasium as gym
from ray.rllib.algorithms.algorithm import Algorithm
import os
import sys
from copy import deepcopy

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.rllib_single_agent_wrapper import RLLibSingleAgentWrapper
from src.env.peer_group_environment import PeerGroupEnvironment
from ray import tune


In [ ]:
# 1. Load RLlib PPO checkpoints
checkpoint_paths = [
    Path("../checkpoints/PPO_checkpoints/ppo_balanced_by_effort_iter0099_mrl50_djwnaafk_11-05-09-06_eval_na_periodic_seed1").resolve(),
    Path("../checkpoints/PPO_checkpoints/ppo_balanced_by_effort_iter0099_mrl50_gvr49cx9_12-05-02-40_eval_na_periodic_seed2").resolve(),
    Path("../checkpoints/PPO_checkpoints/ppo_balanced_by_effort_iter0099_mrl50_064xj18g_12-05-16-21_eval_na_periodic_seed3").resolve(),
    Path("../checkpoints/PPO_checkpoints/ppo_balanced_by_effort_iter0099_mrl50_szarbkte_13-05-20-23_eval_na_periodic_seed4").resolve(),
    Path("../checkpoints/PPO_checkpoints/ppo_balanced_by_effort_iter0099_mrl50_vsy7vsv9_14-05-07-22_eval_na_periodic_seed5").resolve()
]

for checkpoint_path in checkpoint_paths:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint path {checkpoint_path} not found.")

# Initialize Ray if not already
if not ray.is_initialized():
    ray.init(ignore_reinit_error=True, local_mode=True)

# Training configuration values
env_config = {
    "start_agents": 100,
    "max_agents": 400,
    "max_steps": 600,
    "max_peer_group_size": 40,
    "n_groups": 10,
    "n_projects_per_step": 1,
    "max_projects_per_agent": 8,
    "reward_mode": "by_effort",
}

ENV_NAME = "peer_group_single_agent_fixed_population"

def env_creator(config):
    env = PeerGroupEnvironment(
        start_agents=env_config["start_agents"],
        max_agents=env_config["max_agents"],
        max_steps=env_config["max_steps"],
        max_peer_group_size=env_config["max_peer_group_size"],
        n_groups=env_config["n_groups"],
        n_projects_per_step=env_config["n_projects_per_step"],
        max_projects_per_agent=env_config["max_projects_per_agent"],
        reward_mode=env_config["reward_mode"],
    )
    return RLLibSingleAgentWrapper(env)

tune.register_env(ENV_NAME, env_creator)

# Create wrapper instance for feature index map
dummy_env = PeerGroupEnvironment(
    start_agents=env_config["start_agents"],
    max_agents=env_config["max_agents"],
    max_steps=env_config["max_steps"],
    max_peer_group_size=env_config["max_peer_group_size"],
    n_groups=env_config["n_groups"],
    n_projects_per_step=env_config["n_projects_per_step"],
    max_projects_per_agent=env_config["max_projects_per_agent"],
    reward_mode=env_config["reward_mode"]
)
wrapper_instance = RLLibSingleAgentWrapper(dummy_env)

# Build feature index map
feature_index = wrapper_instance.get_feature_index_map()
print(f"Feature index map built with {len(feature_index)} features.")

# Load all checkpoints
algos = []
policies = []

for i, checkpoint_path in enumerate(checkpoint_paths, 1):
    try:
        algo = Algorithm.from_checkpoint(str(checkpoint_path))
        print(f"Checkpoint {i}/5 restored successfully from {checkpoint_path.name}")
        algos.append(algo)

        policy = algo.get_policy()
        policies.append(policy)

        if i == 1:
            print(f"Policy Observation Space: {policy.observation_space}")
            print(f"Policy Action Space: {policy.action_space}")
            expected_obs_size = policy.observation_space.shape[0]
            print(f"Expected observation size: {expected_obs_size}")
    except Exception as e:
        print(f"Failed to load checkpoint {i}: {e}")
        raise e

In [ ]:
# 2. Generate fresh observations for each checkpoint
print("\n--- Generating fresh observations for all 5 checkpoints ---")

all_checkpoint_observations = []  # List of (checkpoint_idx, observations) tuples
seeds = range(501, 521)
max_obs_per_checkpoint = 1000

for checkpoint_idx, algo in enumerate(algos, 1):
    print(f"\nCheckpoint {checkpoint_idx}/5:")
    fresh_observations = []
    
    for seed in seeds:
        obs, info = wrapper_instance.reset(seed=seed)
        fresh_observations.append(obs)
        
        terminated = truncated = False
        while not (terminated or truncated):
            # Use policy to get action
            action = algo.compute_single_action(obs, policy_id="default_policy")
            obs, reward, terminated, truncated, info = wrapper_instance.step(action)
            
            assert obs.shape[0] == expected_obs_size, f"Obs size mismatch: {obs.shape[0]} != {expected_obs_size}"
            fresh_observations.append(obs)
            
            if len(fresh_observations) >= max_obs_per_checkpoint:
                break
        if len(fresh_observations) >= max_obs_per_checkpoint:
            break
    
    print(f"  Collected {len(fresh_observations)} observations")
    all_checkpoint_observations.append((checkpoint_idx, fresh_observations))

total_obs = sum(len(obs_list) for _, obs_list in all_checkpoint_observations)
print(f"\nTotal observations collected: {total_obs}")

In [ ]:
# 3. Corrected flat-vector sensitivity functions

def get_head_slices(action_space):
    """
    Build correct logit slices for a Gymnasium Dict action space.

    Important fix:
    - Discrete(n) uses n logits.
    - MultiDiscrete(nvec) uses sum(nvec) logits, not prod(shape).

    Example:
    MultiDiscrete([2] * 40) represents 40 binary categorical decisions.
    It therefore needs 80 logits: 2 logits per peer.
    """
    if not isinstance(action_space, gym.spaces.Dict):
        raise TypeError(f"Expected gym.spaces.Dict action space, got {type(action_space)}: {action_space}")

    slices = {}
    start = 0

    # Preserve the action-space order defined by the wrapper/RLlib.
    for head, space in action_space.spaces.items():
        if isinstance(space, gym.spaces.Discrete):
            size = int(space.n)
        elif isinstance(space, gym.spaces.MultiDiscrete):
            size = int(np.sum(space.nvec))
        else:
            raise NotImplementedError(f"Unsupported action space for head '{head}': {space}")

        slices[head] = {
            "start": start,
            "end": start + size,
            "size": size,
            "space": space,
        }
        start += size

    return slices


def _probabilities_from_head_logits(head_logits, space):
    """
    Convert one action-head logit vector into probabilities used for sensitivity.

    For Discrete heads, the full softmax probability vector is used.
    For MultiDiscrete heads, logits are split into one categorical distribution
    per sub-action. For binary sub-actions, only P(action=1) is used because
    this directly represents the collaboration intent probability.
    """
    if isinstance(space, gym.spaces.Discrete):
        return F.softmax(head_logits, dim=0)

    if isinstance(space, gym.spaces.MultiDiscrete):
        sizes = [int(n) for n in space.nvec]
        parts = torch.split(head_logits, sizes)

        if len(parts) != len(sizes):
            raise ValueError(
                f"MultiDiscrete split failed: expected {len(sizes)} parts, got {len(parts)}. "
                f"sizes={sizes}, logits={head_logits.shape}"
            )

        probs = []
        for part in parts:
            p = F.softmax(part, dim=0)

            # Binary sub-action: use probability of action 1.
            if part.shape[0] == 2:
                probs.append(p[1].reshape(1))
            else:
                # Generic fallback for non-binary MultiDiscrete sub-actions.
                probs.append(p.reshape(-1))

        return torch.cat(probs)

    raise NotImplementedError(f"Unsupported action space: {space}")


def diagnose_logit_splitting(policy, obs_vec):
    """
    Return diagnostics that verify whether the action-space logit split matches
    the actual policy output size.
    """
    model = policy.model
    model.eval()

    obs_tensor = torch.tensor(obs_vec, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        logits, _ = model({"obs": obs_tensor}, [], torch.tensor([1]))

    head_slices = get_head_slices(policy.action_space)
    expected_logits = max(meta["end"] for meta in head_slices.values())
    actual_logits = int(logits.shape[-1])

    return {
        "action_space": policy.action_space,
        "head_slices": head_slices,
        "expected_logits": expected_logits,
        "actual_logits": actual_logits,
        "logits_shape": tuple(logits.shape),
        "is_valid": expected_logits == actual_logits,
    }


def compute_flat_logit_sensitivity(policy, obs_vec, feature_idx, epsilon=1e-3):
    """
    Compute local policy sensitivity to a small perturbation in one observation feature.

    The metric is mean absolute probability change for each action head.
    This is a local policy-output sensitivity, not a causal reward effect.
    """
    model = policy.model
    model.eval()

    obs_orig = torch.tensor(obs_vec, dtype=torch.float32).unsqueeze(0)
    obs_pert = obs_orig.clone()
    obs_pert[0, feature_idx] += epsilon

    with torch.no_grad():
        logits_orig, _ = model({"obs": obs_orig}, [], torch.tensor([1]))
        logits_pert, _ = model({"obs": obs_pert}, [], torch.tensor([1]))

    head_slices = get_head_slices(policy.action_space)
    expected_logits = max(meta["end"] for meta in head_slices.values())
    actual_logits = int(logits_orig.shape[-1])

    if expected_logits != actual_logits:
        formatted_slices = {
            head: {"start": meta["start"], "end": meta["end"], "size": meta["size"], "space": str(meta["space"])}
            for head, meta in head_slices.items()
        }
        raise ValueError(
            f"Logit split mismatch: expected {expected_logits}, got {actual_logits}. "
            f"Computed head slices: {formatted_slices}"
        )

    results = {}
    for head, meta in head_slices.items():
        start = meta["start"]
        end = meta["end"]
        space = meta["space"]

        l_orig = logits_orig[0, start:end]
        l_pert = logits_pert[0, start:end]

        p_orig = _probabilities_from_head_logits(l_orig, space)
        p_pert = _probabilities_from_head_logits(l_pert, space)

        delta_probs = p_pert - p_orig
        results[head] = {
            "mean_abs_delta_prob": torch.mean(torch.abs(delta_probs)).item(),
            "n_probabilities": int(p_orig.numel()),
        }

    return results


In [ ]:
# 4. Dynamically build feature list based on active slots in observations

def build_dynamic_feature_list(observations, feature_index, max_projects=8, max_peers=40):
    """
    Analyze observations to determine which slots are actually active,
    and build a feature list that only includes active slots.
    """
    # Always include global features
    features_to_analyze = {
        "age": "observation.age",
        "accumulated_rewards": "observation.accumulated_rewards",
    }
    
    # Track which project opportunity slots exist in the observation encoding.
    project_opp_active = set()
    for obs in observations:
        for i in range(10):  # Check up to 10 potential opportunity slots.
            novelty_key = f"observation.project_opportunities.project_{i}.novelty"
            if novelty_key in feature_index:
                idx = feature_index[novelty_key]
                if idx < len(obs):
                    project_opp_active.add(i)
    
    # Track which running project and peer slots are actually used.
    running_active_counts = {}
    peer_active_counts = {}
    
    for obs in observations:
        for i in range(max_projects):
            is_active_key = f"observation.running_projects.project_{i}.is_active"
            if is_active_key in feature_index:
                idx = feature_index[is_active_key]
                if idx < len(obs) and obs[idx] > 0.5:
                    running_active_counts[i] = running_active_counts.get(i, 0) + 1
        
        for i in range(max_peers):
            peer_key = f"observation.peer_group.{i}"
            if peer_key in feature_index:
                idx = feature_index[peer_key]
                if idx < len(obs) and obs[idx] > 0.5:
                    peer_active_counts[i] = peer_active_counts.get(i, 0) + 1
    
    # Add project opportunity features.
    for i in sorted(project_opp_active):
        features_to_analyze[f"project_{i} novelty"] = f"observation.project_opportunities.project_{i}.novelty"
        features_to_analyze[f"project_{i} prestige"] = f"observation.project_opportunities.project_{i}.prestige"
        features_to_analyze[f"project_{i} effort"] = f"observation.project_opportunities.project_{i}.required_effort"
        features_to_analyze[f"project_{i} time"] = f"observation.project_opportunities.project_{i}.time_window"
    
    # Add running project features only for slots active in more than 5% of observations.
    threshold = len(observations) * 0.05
    for i in sorted(running_active_counts.keys()):
        if running_active_counts[i] > threshold:
            features_to_analyze[f"running_{i} effort"] = f"observation.running_projects.project_{i}.current_effort"
            features_to_analyze[f"running_{i} time_left"] = f"observation.running_projects.project_{i}.time_left"
            features_to_analyze[f"running_{i} is_active"] = f"observation.running_projects.project_{i}.is_active"
    
    # Add peer features only for peer slots active in more than 5% of observations.
    for i in sorted(peer_active_counts.keys()):
        if peer_active_counts[i] > threshold:
            features_to_analyze[f"peer_{i} active"] = f"observation.peer_group.{i}"
            features_to_analyze[f"peer_{i} reputation"] = f"observation.peer_reputation.{i}"
    
    return features_to_analyze, project_opp_active, running_active_counts, peer_active_counts


print("\n--- Verifying corrected logit splitting before sensitivity analysis ---")
first_policy = policies[0]
first_obs = all_checkpoint_observations[0][1][0]
diagnostics = diagnose_logit_splitting(first_policy, first_obs)
print(f"Expected logits: {diagnostics['expected_logits']}")
print(f"Actual logits:   {diagnostics['actual_logits']}")
if not diagnostics["is_valid"]:
    raise ValueError(f"Invalid logit split diagnostics: {diagnostics}")
print("Logit split validation passed.")

print("\n--- Building dynamic feature list from all observations ---")
all_observations_flat = []
for checkpoint_idx, obs_list in all_checkpoint_observations:
    all_observations_flat.extend(obs_list)

features_to_analyze, proj_opp_active, run_active, peer_active = build_dynamic_feature_list(
    all_observations_flat, feature_index
)

print(f"\nActive project opportunities: {sorted(proj_opp_active)}")
print(f"Active running project slots (>5% obs): {sorted([k for k, v in run_active.items() if v > len(all_observations_flat) * 0.05])}")
print(f"Active peer slots (>5% obs): {len([k for k, v in peer_active.items() if v > len(all_observations_flat) * 0.05])}")
print(f"\nTotal features to analyze: {len(features_to_analyze)}")

# Show feature breakdown.
print("\nFeature breakdown:")
global_features = [k for k in features_to_analyze.keys() if k in ["age", "accumulated_rewards"]]
proj_features = [k for k in features_to_analyze.keys() if k.startswith("project_")]
run_features = [k for k in features_to_analyze.keys() if k.startswith("running_")]
peer_features = [k for k in features_to_analyze.keys() if k.startswith("peer_")]
print(f"  Global: {len(global_features)}")
print(f"  Project opportunities: {len(proj_features)}")
print(f"  Running projects: {len(run_features)}")
print(f"  Peers: {len(peer_features)}")

# 5. Run Sensitivity Analysis for all checkpoints with dynamic features.
print("\n--- Running Sensitivity Analysis for all checkpoints ---")
sensitivity_results = []
epsilon = 1e-3

for checkpoint_idx, fresh_observations in all_checkpoint_observations:
    policy = policies[checkpoint_idx - 1]
    print(f"\nCheckpoint {checkpoint_idx}/5: Analyzing {len(features_to_analyze)} features...")
    
    feature_count = 0
    for label, feature_name in features_to_analyze.items():
        if feature_name not in feature_index:
            print(f"  Warning: {feature_name} not found in feature index map. Skipping.")
            continue
            
        idx = feature_index[feature_name]
        feature_count += 1
        
        if feature_count % 10 == 0 or feature_count == len(features_to_analyze):
            print(f"  Progress: {feature_count}/{len(features_to_analyze)} features...")
        
        for obs_id, obs in enumerate(fresh_observations):
            res = compute_flat_logit_sensitivity(policy, obs, idx, epsilon=epsilon)
            for head, metrics in res.items():
                sensitivity_results.append({
                    "checkpoint": checkpoint_idx,
                    "observation_id": obs_id,
                    "feature": label,
                    "head": head,
                    "sensitivity": metrics["mean_abs_delta_prob"] / epsilon,
                    "n_probabilities": metrics["n_probabilities"],
                })

df_sensitivity = pd.DataFrame(sensitivity_results)

# 6. Aggregate results.
# First average over observations per checkpoint. Then compute mean/std across checkpoints.
# This keeps observation-level variation separate from checkpoint/seed variation.
print("\n--- Aggregating results ---")
checkpoint_sensitivity = (
    df_sensitivity
    .groupby(["checkpoint", "feature", "head"], as_index=False)["sensitivity"]
    .mean()
)

agg_sensitivity_all = checkpoint_sensitivity.groupby(["feature", "head"])["sensitivity"].mean().unstack()
agg_sensitivity_std = checkpoint_sensitivity.groupby(["feature", "head"])["sensitivity"].std().unstack()

print("\nTop 15 features by mean sensitivity (across all action heads):")
overall_mean = agg_sensitivity_all.mean(axis=1).sort_values(ascending=False)
print(overall_mean.head(15))


In [ ]:
# 6. Visualization: one separate plot per head (aggregated across checkpoints)
heads = agg_sensitivity_all.columns

for head in heads:
    plt.figure(figsize=(10, 5))

    agg_sensitivity_all[head] \
        .sort_values(ascending=False) \
        .plot(kind="bar")

    plt.title(f"Policy Sensitivity: {head} (mean across checkpoints)")
    plt.ylabel("mean(|Δprob|) / epsilon")
    plt.xlabel("Feature")
    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

# Comparison plot showing variation across checkpoints.
print("\n--- Variation analysis across checkpoints ---")
print("\nStandard deviation of checkpoint-level mean sensitivity:")
print(agg_sensitivity_std)

for head in heads:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Mean sensitivity across checkpoints.
    agg_sensitivity_all[head] \
        .sort_values(ascending=False) \
        .plot(kind="bar", ax=ax1)
    ax1.set_title(f"Mean Sensitivity: {head}")
    ax1.set_ylabel("mean(|Δprob|) / epsilon")
    ax1.set_xlabel("Feature")
    ax1.tick_params(axis="x", rotation=45)
    
    # Standard deviation across checkpoint-level means.
    feature_order = agg_sensitivity_all[head].sort_values(ascending=False).index
    agg_sensitivity_std[head] \
        .reindex(feature_order) \
        .plot(kind="bar", ax=ax2)
    ax2.set_title(f"Std Across Checkpoints: {head}")
    ax2.set_ylabel("std checkpoint mean sensitivity")
    ax2.set_xlabel("Feature")
    ax2.tick_params(axis="x", rotation=45)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# 7. Comprehensive tabular results for all features
print("=" * 80)
print("COMPREHENSIVE SENSITIVITY RESULTS - CORRECTED LOGIT SPLIT")
print("=" * 80)

# Statistics are computed on checkpoint-level means, not directly on all observations.
summary_table = checkpoint_sensitivity.groupby(["feature", "head"]).agg({
    "sensitivity": ["mean", "std", "min", "max", "count"]
}).round(6)

summary_table.columns = ["Mean", "Std Dev Across Checkpoints", "Min", "Max", "N_checkpoints"]
print("\nDetailed statistics per feature and action head:")
print(summary_table)

# Pivot table: Mean sensitivity.
print("\n" + "=" * 80)
print("MEAN SENSITIVITY (mean across checkpoint-level means)")
print("=" * 80)
pivot_mean = agg_sensitivity_all.round(6)
print(pivot_mean)

# Pivot table: Std Dev across checkpoints.
print("\n" + "=" * 80)
print("STANDARD DEVIATION (across checkpoint-level means)")
print("=" * 80)
pivot_std = agg_sensitivity_std.round(6)
print(pivot_std)

# Overall ranking by mean sensitivity aggregated across all heads.
print("\n" + "=" * 80)
print("OVERALL FEATURE RANKING (mean across all action heads)")
print("=" * 80)
overall_ranking = pivot_mean.mean(axis=1).sort_values(ascending=False)
print(overall_ranking.round(6))

# Per action head ranking.
print("\n" + "=" * 80)
print("TOP 5 FEATURES PER ACTION HEAD")
print("=" * 80)
for head in heads:
    print(f"\n{head.upper()}:")
    top5 = pivot_mean[head].sort_values(ascending=False).head(5)
    for i, (feature, value) in enumerate(top5.items(), 1):
        print(f"  {i}. {feature:30s} {value:.6f}")

# Export to DataFrame for thesis tables / further analysis.
print("\n" + "=" * 80)
print("SUMMARY TABLE (sorted by mean sensitivity)")
print("=" * 80)

def _safe_values(df, column):
    if column in df.columns:
        return df[column].values
    return np.full(len(df.index), np.nan)

summary_export = pd.DataFrame({
    "Feature": pivot_mean.index,
    "choose_project_mean": _safe_values(pivot_mean, "choose_project"),
    "choose_project_std": _safe_values(pivot_std, "choose_project"),
    "collaborate_with_mean": _safe_values(pivot_mean, "collaborate_with"),
    "collaborate_with_std": _safe_values(pivot_std, "collaborate_with"),
    "put_effort_mean": _safe_values(pivot_mean, "put_effort"),
    "put_effort_std": _safe_values(pivot_std, "put_effort"),
})
summary_export["overall_mean"] = summary_export[[
    "choose_project_mean", "collaborate_with_mean", "put_effort_mean"
]].mean(axis=1)
summary_export = summary_export.sort_values("overall_mean", ascending=False)
print(summary_export.to_string(index=False))

# Save corrected CSV without overwriting the original result file.
output_path = Path("sensitivity_analysis_results_fixed.csv")
summary_export.to_csv(output_path, index=False)
print(f"\nResults saved to {output_path.resolve()}")


In [ ]:
# 8. Diagnostic output - verification of corrected logit splitting
print("=" * 80)
print("DIAGNOSTIC OUTPUT - VERIFICATION OF LOGIT SPLITTING FIX")
print("=" * 80)

policy = policies[0]
obs_vec = all_checkpoint_observations[0][1][0]
diagnostics = diagnose_logit_splitting(policy, obs_vec)
act_space = diagnostics["action_space"]
head_slices = diagnostics["head_slices"]

print("\n1. ACTION SPACE:")
print(f"   Type: {type(act_space)}")
print(f"   Space: {act_space}")
for head, space in act_space.spaces.items():
    print(f"   - {head}: {space}")
    if isinstance(space, gym.spaces.Discrete):
        print(f"     logits: {space.n}")
    elif isinstance(space, gym.spaces.MultiDiscrete):
        print(f"     nvec length: {len(space.nvec)}")
        print(f"     nvec sum: {int(np.sum(space.nvec))}")
        print(f"     explanation: {len(space.nvec)} sub-actions with categorical sizes {set(space.nvec.tolist())}")

print("\n2. COMPUTED LOGIT SLICES PER ACTION HEAD:")
for head, meta in head_slices.items():
    print(f"   - {head}:")
    print(f"     start={meta['start']}, end={meta['end']}, size={meta['size']}")
    print(f"     space={meta['space']}")

print("\n3. VALIDATION:")
print(f"   Total expected logits: {diagnostics['expected_logits']}")
print(f"   Actual logits shape:   {diagnostics['logits_shape']}")
print(f"   Actual logits count:   {diagnostics['actual_logits']}")
print(f"   Valid split:           {diagnostics['is_valid']}")

if not diagnostics["is_valid"]:
    raise ValueError("Logit split is still invalid. Do not use the resulting CSV.")

print("\n4. OUTPUT:")
print(f"   Corrected CSV: {Path('sensitivity_analysis_results_fixed.csv').resolve()}")
print("\nThe notebook uses sum(space.nvec) for MultiDiscrete heads and softmax per sub-action.")
